# Summarization Model — Fine-tuning T5 on Chat Conversations

**Abstractive summarization** using a fine-tuned **T5-small** model for chat conversation summaries.

**What this notebook does:**
1. Loads the **SAMSum** dataset (chat → summary pairs)
2. Preprocesses conversations for T5 input format
3. Fine-tunes **T5-small** using Hugging Face `Trainer`
4. Evaluates with ROUGE metrics
5. Tests with sample chat conversations
6. Exports the model for backend inference

**Runtime:** Use GPU if available (Kaggle / Colab). Falls back to CPU automatically.

In [ ]:
# ============================================================
# Cell 1 — Install dependencies (run once)
# ============================================================
%pip install -q torch transformers datasets evaluate rouge-score sentencepiece accelerate

In [ ]:
# ============================================================
# Cell 2 — Imports & Device Setup
# ============================================================
import torch
import numpy as np
import os
import json
import random
from pathlib import Path
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import evaluate

# Auto-detect GPU / CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 1. Dataset — SAMSum

**SAMSum** (Samsung Summarization) contains ~16k messenger-like conversations with human-written summaries.  
This is the best publicly available dataset for **chat summarization** — the conversations look like real chat messages.

Each example: **(dialogue, summary)**

In [ ]:
# ============================================================
# Cell 3 — Load SAMSum dataset
# ============================================================
dataset = load_dataset("samsum")

print(f"Train examples: {len(dataset['train'])}")
print(f"Val examples:   {len(dataset['validation'])}")
print(f"Test examples:  {len(dataset['test'])}")

# Show a sample
sample = dataset['train'][0]
print(f"\n{'='*60}")
print(f"DIALOGUE:")
print(sample['dialogue'])
print(f"\nSUMMARY:")
print(sample['summary'])
print(f"{'='*60}")

In [ ]:
# ============================================================
# Cell 4 — Explore dataset statistics
# ============================================================
dialogue_lengths = [len(ex['dialogue'].split()) for ex in dataset['train']]
summary_lengths = [len(ex['summary'].split()) for ex in dataset['train']]

print(f"Dialogue word counts — Mean: {np.mean(dialogue_lengths):.0f}, "
      f"Median: {np.median(dialogue_lengths):.0f}, "
      f"Max: {np.max(dialogue_lengths)}, "
      f"95th pct: {np.percentile(dialogue_lengths, 95):.0f}")
print(f"Summary word counts  — Mean: {np.mean(summary_lengths):.0f}, "
      f"Median: {np.median(summary_lengths):.0f}, "
      f"Max: {np.max(summary_lengths)}, "
      f"95th pct: {np.percentile(summary_lengths, 95):.0f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(dialogue_lengths, bins=50, color='#7c3aed', alpha=0.7, edgecolor='white')
axes[0].set_title('Dialogue Length Distribution (words)')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].axvline(np.percentile(dialogue_lengths, 95), color='red', linestyle='--', label='95th pct')
axes[0].legend()

axes[1].hist(summary_lengths, bins=50, color='#06b6d4', alpha=0.7, edgecolor='white')
axes[1].set_title('Summary Length Distribution (words)')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].axvline(np.percentile(summary_lengths, 95), color='red', linestyle='--', label='95th pct')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Tokenization & Preprocessing

We use **T5-small** with a `"summarize: "` prefix.  
- Max input length: **512 tokens** (covers 95%+ of dialogues)
- Max target length: **128 tokens** (covers all summaries)

In [ ]:
# ============================================================
# Cell 5 — Load T5 tokenizer and define preprocessing
# ============================================================
MODEL_NAME = "t5-small"
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128
PREFIX = "summarize: "

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, legacy=False)
print(f"Tokenizer: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")


def preprocess_function(examples):
    """Tokenize dialogues and summaries for T5."""
    inputs = [PREFIX + dialogue for dialogue in examples['dialogue']]
    targets = examples['summary']

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,  # Dynamic padding via data collator
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )

    model_inputs['labels'] = labels['input_ids']
    return model_inputs


# Show tokenization example
sample_input = PREFIX + dataset['train'][0]['dialogue']
sample_tokens = tokenizer(sample_input, truncation=True, max_length=MAX_INPUT_LENGTH)
print(f"\nSample tokenized input length: {len(sample_tokens['input_ids'])} tokens")
print(f"First 20 tokens decoded: {tokenizer.decode(sample_tokens['input_ids'][:20])}")

In [ ]:
# ============================================================
# Cell 6 — Tokenize the entire dataset
# ============================================================
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset['train'].column_names,
    desc="Tokenizing",
)

print(f"Tokenized train: {len(tokenized_dataset['train'])} examples")
print(f"Tokenized val:   {len(tokenized_dataset['validation'])} examples")
print(f"Tokenized test:  {len(tokenized_dataset['test'])} examples")
print(f"\nFeatures: {tokenized_dataset['train'].features}")

## 3. Model & Training Setup

Fine-tune **T5-small** (60M parameters) using Hugging Face `Seq2SeqTrainer`.  
Training config:
- **Epochs:** 5
- **Batch size:** 8 (with gradient accumulation = 2 → effective batch 16)
- **Learning rate:** 3e-4 with linear warmup
- **FP16:** Enabled on GPU for faster training
- **Evaluation:** Every epoch with ROUGE metrics

In [ ]:
# ============================================================
# Cell 7 — Load model and define ROUGE metric
# ============================================================
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
print(f"Model: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Data collator for dynamic padding
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

# ROUGE metric
rouge_metric = evaluate.load("rouge")


def compute_metrics(eval_preds):
    """Compute ROUGE scores for evaluation."""
    preds, labels = eval_preds

    # Replace -100 in labels (padding) with pad_token_id
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Strip whitespace
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    # Compute ROUGE
    result = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )

    return {k: round(v * 100, 2) for k, v in result.items()}


print("ROUGE metric loaded.")

In [ ]:
# ============================================================
# Cell 8 — Define training arguments
# ============================================================
OUTPUT_DIR = "./t5_summarization_output"
NUM_EPOCHS = 5
BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 2  # Effective batch size = 16
LEARNING_RATE = 3e-4
WARMUP_STEPS = 200
WEIGHT_DECAY = 0.01

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
)

print(f"Training config:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE} x {GRADIENT_ACCUMULATION} accumulation = {BATCH_SIZE * GRADIENT_ACCUMULATION} effective")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  FP16: {training_args.fp16}")
print(f"  Output: {OUTPUT_DIR}")

In [ ]:
# ============================================================
# Cell 9 — Initialize Trainer and start training
# ============================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f"Starting training...")
print(f"  Train examples: {len(tokenized_dataset['train'])}")
print(f"  Val examples:   {len(tokenized_dataset['validation'])}")
print(f"  Steps per epoch: {len(tokenized_dataset['train']) // (BATCH_SIZE * GRADIENT_ACCUMULATION)}")
print()

train_result = trainer.train()

# Print training summary
print(f"\n{'='*60}")
print(f"Training complete!")
print(f"  Total steps: {train_result.global_step}")
print(f"  Training loss: {train_result.training_loss:.4f}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# Cell 10 — Plot training curves
# ============================================================
log_history = trainer.state.log_history

# Extract training loss
train_steps = [entry['step'] for entry in log_history if 'loss' in entry]
train_losses = [entry['loss'] for entry in log_history if 'loss' in entry]

# Extract eval metrics
eval_epochs = [entry['epoch'] for entry in log_history if 'eval_loss' in entry]
eval_losses = [entry['eval_loss'] for entry in log_history if 'eval_loss' in entry]
eval_rouge1 = [entry.get('eval_rouge1', 0) for entry in log_history if 'eval_loss' in entry]
eval_rouge2 = [entry.get('eval_rouge2', 0) for entry in log_history if 'eval_loss' in entry]
eval_rougeL = [entry.get('eval_rougeL', 0) for entry in log_history if 'eval_loss' in entry]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training loss
axes[0].plot(train_steps, train_losses, color='#7c3aed', alpha=0.8, linewidth=1.5)
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

# Eval loss
axes[1].plot(eval_epochs, eval_losses, 'o-', color='#ef4444', linewidth=2, markersize=8)
axes[1].set_title('Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].grid(True, alpha=0.3)

# ROUGE scores
axes[2].plot(eval_epochs, eval_rouge1, 'o-', label='ROUGE-1', color='#06b6d4', linewidth=2, markersize=8)
axes[2].plot(eval_epochs, eval_rouge2, 's-', label='ROUGE-2', color='#7c3aed', linewidth=2, markersize=8)
axes[2].plot(eval_epochs, eval_rougeL, '^-', label='ROUGE-L', color='#f59e0b', linewidth=2, markersize=8)
axes[2].set_title('ROUGE Scores (Validation)')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Score')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final metrics
if eval_rougeL:
    print(f"\nBest ROUGE-L: {max(eval_rougeL):.2f}")
    print(f"Best ROUGE-1: {max(eval_rouge1):.2f}")
    print(f"Best ROUGE-2: {max(eval_rouge2):.2f}")

## 4. Evaluation on Test Set

In [ ]:
# ============================================================
# Cell 11 — Evaluate on test set
# ============================================================
test_results = trainer.evaluate(tokenized_dataset['test'])

print(f"{'='*60}")
print(f"Test Set Results:")
print(f"{'='*60}")
for key, value in sorted(test_results.items()):
    if key.startswith('eval_'):
        name = key.replace('eval_', '').upper()
        print(f"  {name}: {value:.2f}" if isinstance(value, float) else f"  {name}: {value}")
print(f"{'='*60}")

## 5. Sample Predictions

In [ ]:
# ============================================================
# Cell 12 — Generate summaries for sample dialogues
# ============================================================
model.eval()
model.to(device)


def generate_summary(dialogue, max_length=128, num_beams=4, length_penalty=1.0):
    """Generate a summary for a chat dialogue."""
    input_text = PREFIX + dialogue
    inputs = tokenizer(
        input_text,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=num_beams,
            length_penalty=length_penalty,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


# Test on SAMSum test set examples
print("=" * 70)
print("Sample Predictions from Test Set")
print("=" * 70)

for i in range(5):
    example = dataset['test'][i]
    predicted = generate_summary(example['dialogue'])

    print(f"\n--- Example {i+1} ---")
    print(f"DIALOGUE:\n{example['dialogue']}")
    print(f"\nREFERENCE SUMMARY: {example['summary']}")
    print(f"PREDICTED SUMMARY: {predicted}")
    print("-" * 70)

In [ ]:
# ============================================================
# Cell 13 — Test with custom chat conversations
# ============================================================
custom_dialogues = [
    # Project planning
    """Alice: Hey team, we need to finalize the project timeline.
Bob: I think we can finish the backend by Friday.
Alice: What about the frontend?
Charlie: I need at least 2 more weeks for the UI.
Alice: Okay, let's set the deadline for March 15th then.
Bob: Works for me. I'll update the project board.
Charlie: Sounds good. I'll start on the dashboard component today.""",

    # Casual conversation
    """John: Did you watch the game last night?
Sarah: Yes! It was amazing. The final score was 3-2.
John: I can't believe they scored in the last minute.
Sarah: I know, right? That goal was incredible.
John: We should go watch the next game together.
Sarah: Definitely! Let me check the schedule.""",

    # Meeting scheduling
    """Emma: Can we reschedule our meeting?
David: Sure, when works for you?
Emma: How about Thursday at 2pm?
David: I have a call at 2. Could we do 3pm instead?
Emma: 3pm works perfectly. I'll send the calendar invite.
David: Great, see you then!""",

    # Bug discussion
    """Dev1: Found a critical bug in the payment module.
Dev2: What's happening?
Dev1: The checkout fails when users apply a discount code.
Dev2: That sounds like the validation logic. Let me check.
Dev1: I traced it to the price calculation function.
Dev2: Found it! The discount was applied twice. Pushing the fix now.
Dev1: Great, let me know when it's deployed so I can test.""",
]

print("=" * 70)
print("Custom Chat Conversation Summaries")
print("=" * 70)

for i, dialogue in enumerate(custom_dialogues):
    predicted = generate_summary(dialogue)
    print(f"\n--- Conversation {i+1} ---")
    print(f"DIALOGUE:\n{dialogue}")
    print(f"\nSUMMARY: {predicted}")
    print("-" * 70)

## 6. Detailed ROUGE Analysis

In [ ]:
# ============================================================
# Cell 14 — Compute detailed ROUGE on full test set
# ============================================================
from tqdm import tqdm

print("Generating predictions on test set...")
predictions = []
references = []

for example in tqdm(dataset['test'], desc="Generating"):
    pred = generate_summary(example['dialogue'])
    predictions.append(pred)
    references.append(example['summary'])

# Compute ROUGE
results = rouge_metric.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True,
)

print(f"\n{'='*60}")
print(f"Full Test Set ROUGE Scores:")
print(f"{'='*60}")
for key, value in results.items():
    print(f"  {key.upper()}: {value * 100:.2f}")
print(f"{'='*60}")

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
names = [k.upper() for k in results.keys()]
values = [v * 100 for v in results.values()]
colors = ['#7c3aed', '#06b6d4', '#f59e0b', '#ef4444']
bars = ax.bar(names, values, color=colors[:len(names)], edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}', ha='center', va='bottom', fontweight='bold')

ax.set_title('ROUGE Scores on Test Set', fontsize=14, fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, max(values) + 10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 7. Export Model for Backend

Saves the fine-tuned T5 model and tokenizer that the FastAPI backend can load from `ai/saved_models/summarization/`.

In [ ]:
# ============================================================
# Cell 15 — Save model & tokenizer for backend inference
# ============================================================
import shutil

EXPORT_DIR = "/kaggle/working/summarization_export"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Save model and tokenizer
model.cpu()
model.save_pretrained(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

# List exported files
print(f"Exported to '{EXPORT_DIR}/'")
total_size = 0
for f in sorted(os.listdir(EXPORT_DIR)):
    fpath = os.path.join(EXPORT_DIR, f)
    size = os.path.getsize(fpath) / 1e6
    total_size += size
    print(f"  {f} — {size:.1f} MB")
print(f"  Total: {total_size:.1f} MB")

print(f"\nDownload these files from Kaggle Output tab and place them in:")
print(f"  ai/saved_models/summarization/")

In [ ]:
# ============================================================
# Cell 16 — Verify exported model loads correctly
# ============================================================
print("Verifying exported model...")

# Load from export dir
test_tokenizer = T5Tokenizer.from_pretrained(EXPORT_DIR, legacy=False)
test_model = T5ForConditionalGeneration.from_pretrained(EXPORT_DIR)
test_model.eval()

# Test generation
test_dialogue = """Alice: Hey, are you coming to the party tonight?
Bob: What time does it start?
Alice: Around 8pm at my place.
Bob: Sounds fun! I'll be there."""

test_input = test_tokenizer(
    PREFIX + test_dialogue,
    max_length=MAX_INPUT_LENGTH,
    truncation=True,
    return_tensors="pt",
)

with torch.no_grad():
    test_output = test_model.generate(
        **test_input,
        max_length=MAX_TARGET_LENGTH,
        num_beams=4,
        early_stopping=True,
    )

test_summary = test_tokenizer.decode(test_output[0], skip_special_tokens=True)
print(f"\nTest dialogue: {test_dialogue}")
print(f"\nGenerated summary: {test_summary}")
print(f"\nExported model verified successfully!")

# Cleanup
del test_model, test_tokenizer

In [ ]:
# ============================================================
# Cell 17 — Instructions for downloading and deploying
# ============================================================
export_files = os.listdir(EXPORT_DIR)
print("Files in export directory:")
for f in sorted(export_files):
    size = os.path.getsize(os.path.join(EXPORT_DIR, f)) / 1e6
    print(f"  {f} — {size:.2f} MB")

print(f"\n{'='*60}")
print(f"Kaggle / Colab Deployment Instructions")
print(f"{'='*60}")
print(f"1. Go to the 'Output' tab on the right sidebar")
print(f"2. Download ALL files from summarization_export/")
print(f"3. Place them in your project at: ai/saved_models/summarization/")
print(f"4. Required files:")
for f in sorted(export_files):
    print(f"   - {f}")
print(f"5. Restart the backend server — it will auto-load the fine-tuned T5 model")
print(f"{'='*60}")